## Market Data Pipeline
This pipeline notebook will go through several data pages that Prof. Damodaran maintains on [his website](https://pages.stern.nyu.edu/~adamodar/New_Home_Page/home.htm).  Since I can't guarantee that his formatting will be the same, it would be best to run this notebook one cell at a time and ensure that the extracted data matches the proper format.

Damodaran generally updates this data yearly, during the first few weeks of January.

Due to the long lived nature of this data, the inconsistency for when Damodaran will update his webpages, and the potential for formatting changes, it would be best that this pipeline be run manually every year to keep the marketData.json up to date.

The pipeline will edit the marketData.json file in place, so it will need to be added and commited to deploy to the website.

This pipeline was developed using Python 3.12.3.  This pipeline will install the necessary pip packages, so it would be best to use a .venv environment.  A requirements.txt file is included in this folder to better control the requirements.

In [1]:
!pip install requests beautifulsoup4 pandas openpyxl


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import json
from io import BytesIO

import requests
from bs4 import BeautifulSoup
import pandas as pd

In [3]:
# create dummy starting data for market data
market_data = {
  "rfr": 12.34,
  "taxRate": 25,
  "industries": {
    "Choose an Industry": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
  },
  "countries": {
    "Select a Country": {
      "ERP": 12.34
    },
  },
  "regions": {
    "Select a Region": {
      "ERP": 12.34
    },
  },
  "credit_ratings": {
    "Select a Credit Rating": {
      "Spread": 12.34,
      "gt_safe": 100001,
      "gt_risk": 100001,
      "lt_safe": -100001,
      "lt_risk": -100001
    },
  }
}

In [4]:
ctryprem_url = "https://pages.stern.nyu.edu/~adamodar/New_Home_Page/datafile/ctryprem.html"

try:
    response = requests.get(ctryprem_url)
    response.raise_for_status()

    ctryprem_soup = BeautifulSoup(response.text, 'html.parser')

    all_tables = ctryprem_soup.find_all('table')

    ctryprem_table = next(table for table in all_tables if 'Abu Dhabi' in str(table))
    if not ctryprem_table:
        raise ValueError("Could not find the country premium table in the HTML content.")
    
except Exception as e:
    print(f"Error fetching or parsing country premium data: {e}")
    ctryprem_table = None

ctryprem_rows = ctryprem_table.find_all('tr')[1:]
ctryprem_data = {}

for row in ctryprem_rows:
    cols = row.find_all('td')
    country_name = ' '.join(cols[0].get_text(strip=True).split())
    try:
        equity_risk_premium = float(cols[4].get_text(strip=True).replace('%', ''))
    except Exception as e:
        print(f"Invalid ERP value for country {country_name}: {e}")
        equity_risk_premium = 12.34

    try:
        tax_rate = float(cols[5].get_text(strip=True).replace('%', ''))
    except Exception as e:
        print(f"Invalid corp tax rate for country {country_name}: {e}")
        tax_rate = 12.34

    ctryprem_data[country_name] = {
        "ERP": equity_risk_premium,
        "Tax Rate": tax_rate
    }

print(json.dumps(ctryprem_data, indent=2))


Invalid ERP value for country : could not convert string to float: ''
Invalid corp tax rate for country : could not convert string to float: ''
{
  "Abu Dhabi": {
    "ERP": 4.87,
    "Tax Rate": 9.0
  },
  "Albania": {
    "ERP": 8.89,
    "Tax Rate": 15.0
  },
  "Algeria": {
    "ERP": 10.06,
    "Tax Rate": 10.07
  },
  "Andorra (Principality of)": {
    "ERP": 6.3,
    "Tax Rate": 10.0
  },
  "Angola": {
    "ERP": 12.64,
    "Tax Rate": 25.0
  },
  "Argentina": {
    "ERP": 13.94,
    "Tax Rate": 35.0
  },
  "Armenia": {
    "ERP": 8.89,
    "Tax Rate": 18.0
  },
  "Aruba": {
    "ERP": 7.08,
    "Tax Rate": 22.0
  },
  "Australia": {
    "ERP": 4.23,
    "Tax Rate": 30.0
  },
  "Austria": {
    "ERP": 4.59,
    "Tax Rate": 23.0
  },
  "Azerbaijan": {
    "ERP": 7.08,
    "Tax Rate": 20.0
  },
  "Bahamas": {
    "ERP": 10.06,
    "Tax Rate": 0.0
  },
  "Bahrain": {
    "ERP": 11.35,
    "Tax Rate": 0.0
  },
  "Bangladesh": {
    "ERP": 11.35,
    "Tax Rate": 27.5
  },
  "Barbados"

In [5]:
for country in ctryprem_data:
    if country == '':
        continue
    if country not in market_data['countries']:
        print(f"Adding country '{country}' to market data.")
        market_data['countries'][country] = market_data['countries']['Select a Country'].copy()
    market_data['countries'][country]['ERP'] = ctryprem_data[country]['ERP']

print(json.dumps(market_data['countries'], indent=2))

market_data['taxRate'] = ctryprem_data['United States']['Tax Rate']

print('tax rate', json.dumps(market_data['taxRate'], indent=2))

Adding country 'Abu Dhabi' to market data.
Adding country 'Albania' to market data.
Adding country 'Algeria' to market data.
Adding country 'Andorra (Principality of)' to market data.
Adding country 'Angola' to market data.
Adding country 'Argentina' to market data.
Adding country 'Armenia' to market data.
Adding country 'Aruba' to market data.
Adding country 'Australia' to market data.
Adding country 'Austria' to market data.
Adding country 'Azerbaijan' to market data.
Adding country 'Bahamas' to market data.
Adding country 'Bahrain' to market data.
Adding country 'Bangladesh' to market data.
Adding country 'Barbados' to market data.
Adding country 'Belarus' to market data.
Adding country 'Belgium' to market data.
Adding country 'Belize' to market data.
Adding country 'Benin' to market data.
Adding country 'Bermuda' to market data.
Adding country 'Bolivia' to market data.
Adding country 'Bosnia and Herzegovina' to market data.
Adding country 'Botswana' to market data.
Adding country '

In [6]:
psdata_url = "https://pages.stern.nyu.edu/~adamodar/New_Home_Page/datafile/psdata.html"

try:
    response = requests.get(psdata_url)
    response.raise_for_status()

    psdata_soup = BeautifulSoup(response.text, 'html.parser')

    all_tables = psdata_soup.find_all('table')

    psdata_table = next(table for table in all_tables if 'Advertising' in str(table))
    if not psdata_table:
        raise ValueError("Could not find the revenue multiples table in the HTML content.")
    
except Exception as e:
    print(f"Error fetching or parsing revenue multiples data: {e}")
    psdata_table = None

psdata_rows = psdata_table.find_all('tr')[1:]
psdata_data = {}

for row in psdata_rows:
    cols = row.find_all('td')
    industry_name = ' '.join(cols[0].get_text(strip=True).split())
    try:
        ev_sales = float(cols[4].get_text(strip=True).replace('%', ''))
    except Exception as e:
        print(f"Invalid EV/Sales value for industry {industry_name}: {e}")
        ev_sales = None

    try:
        margin = float(cols[5].get_text(strip=True).replace('%', ''))
    except Exception as e:
        print(f"Invalid operating margin for industry {industry_name}: {e}")
        margin = None

    psdata_data[industry_name] = {
        "EV/Sales": ev_sales,
        "Operating Margin": margin
    }

print(json.dumps(psdata_data, indent=2))

Invalid EV/Sales value for industry : could not convert string to float: ''
Invalid operating margin for industry : could not convert string to float: ''
{
  "Advertising": {
    "EV/Sales": 2.12,
    "Operating Margin": 10.05
  },
  "Aerospace/Defense": {
    "EV/Sales": 3.57,
    "Operating Margin": 8.7
  },
  "Air Transport": {
    "EV/Sales": 1.03,
    "Operating Margin": 4.88
  },
  "Apparel": {
    "EV/Sales": 1.59,
    "Operating Margin": 9.89
  },
  "Auto & Truck": {
    "EV/Sales": 3.88,
    "Operating Margin": 2.38
  },
  "Auto Parts": {
    "EV/Sales": 0.82,
    "Operating Margin": 5.8
  },
  "Bank (Money Center)": {
    "EV/Sales": 8.31,
    "Operating Margin": 0.08
  },
  "Banks (Regional)": {
    "EV/Sales": 4.28,
    "Operating Margin": -0.12
  },
  "Beverage (Alcoholic)": {
    "EV/Sales": 2.45,
    "Operating Margin": 22.79
  },
  "Beverage (Soft)": {
    "EV/Sales": 4.16,
    "Operating Margin": 20.63
  },
  "Broadcasting": {
    "EV/Sales": 1.4,
    "Operating Margin

In [7]:
for industry in psdata_data:
    if industry == '':
        continue
    if industry not in market_data['industries']:
        print(f"Adding industry '{industry}' to market data.")
        market_data['industries'][industry] = market_data['industries']['Choose an Industry'].copy()
    market_data['industries'][industry]['EV/Sales'] = psdata_data[industry]['EV/Sales']
    market_data['industries'][industry]['Operating Margin'] = psdata_data[industry]['Operating Margin']

print(json.dumps(market_data['industries'], indent=2))

Adding industry 'Advertising' to market data.
Adding industry 'Aerospace/Defense' to market data.
Adding industry 'Air Transport' to market data.
Adding industry 'Apparel' to market data.
Adding industry 'Auto & Truck' to market data.
Adding industry 'Auto Parts' to market data.
Adding industry 'Bank (Money Center)' to market data.
Adding industry 'Banks (Regional)' to market data.
Adding industry 'Beverage (Alcoholic)' to market data.
Adding industry 'Beverage (Soft)' to market data.
Adding industry 'Broadcasting' to market data.
Adding industry 'Brokerage & Investment Banking' to market data.
Adding industry 'Building Materials' to market data.
Adding industry 'Business & Consumer Services' to market data.
Adding industry 'Cable TV' to market data.
Adding industry 'Chemical (Basic)' to market data.
Adding industry 'Chemical (Diversified)' to market data.
Adding industry 'Chemical (Specialty)' to market data.
Adding industry 'Coal & Related Energy' to market data.
Adding industry 'Com

In [8]:
wacc_url = "https://pages.stern.nyu.edu/~adamodar/New_Home_Page/datafile/wacc.html"

try:
    response = requests.get(wacc_url)
    response.raise_for_status()

    wacc_soup = BeautifulSoup(response.text, 'html.parser')

    all_tables = wacc_soup.find_all('table')

    wacc_table = next(table for table in all_tables if 'Advertising' in str(table))
    if not wacc_table:
        raise ValueError("Could not find the WACC table in the HTML content.")
    
except Exception as e:
    print(f"Error fetching or parsing WACC data: {e}")
    wacc_table = None

wacc_rows = wacc_table.find_all('tr')[1:]
wacc_data = {}

for row in wacc_rows:
    cols = row.find_all('td')
    industry_name = ' '.join(cols[0].get_text(strip=True).split())
    try:
        cod = float(cols[6].get_text(strip=True).replace('%', ''))
    except Exception as e:
        print(f"Invalid cost of debt value for industry {industry_name}: {e}")
        cod = None

    try:
        tax = float(cols[7].get_text(strip=True).replace('%', ''))
    except Exception as e:
        print(f"Invalid tax rate for industry {industry_name}: {e}")
        tax = None

    try:
        coc = float(cols[10].get_text(strip=True).replace('%', ''))
    except Exception as e:
        print(f"Invalid cost of capital for industry {industry_name}: {e}")
        coc = None

    wacc_data[industry_name] = {
        "Cost of Debt": cod,
        "Tax Rate": tax,
        "Cost of Capital": coc
    }

print(json.dumps(wacc_data, indent=2))

Invalid cost of debt value for industry : could not convert string to float: ''
Invalid tax rate for industry : could not convert string to float: ''
Invalid cost of capital for industry : could not convert string to float: ''
{
  "Advertising": {
    "Cost of Debt": 5.29,
    "Tax Rate": 5.02,
    "Cost of Capital": 7.81
  },
  "Aerospace/Defense": {
    "Cost of Debt": 5.29,
    "Tax Rate": 11.58,
    "Cost of Capital": 7.6
  },
  "Air Transport": {
    "Cost of Debt": 5.29,
    "Tax Rate": 8.29,
    "Cost of Capital": 6.72
  },
  "Apparel": {
    "Cost of Debt": 5.29,
    "Tax Rate": 9.61,
    "Cost of Capital": 7.13
  },
  "Auto & Truck": {
    "Cost of Debt": 5.29,
    "Tax Rate": 3.74,
    "Cost of Capital": 9.38
  },
  "Auto Parts": {
    "Cost of Debt": 5.29,
    "Tax Rate": 15.0,
    "Cost of Capital": 8.18
  },
  "Bank (Money Center)": {
    "Cost of Debt": 4.73,
    "Tax Rate": 18.43,
    "Cost of Capital": 4.98
  },
  "Banks (Regional)": {
    "Cost of Debt": 4.73,
    "Tax

In [9]:
for industry in wacc_data:
    if industry == '':
        continue
    if industry not in market_data['industries']:
        print(f"Adding industry '{industry}' to market data.")
        market_data['industries'][industry] = market_data['industries']['Choose an Industry'].copy()
    market_data['industries'][industry]['Cost of Debt'] = wacc_data[industry]['Cost of Debt']
    market_data['industries'][industry]['Tax Rate'] = wacc_data[industry]['Tax Rate']
    market_data['industries'][industry]['Cost of Capital'] = wacc_data[industry]['Cost of Capital']

print(json.dumps(market_data['industries'], indent=2))

{
  "Choose an Industry": {
    "EV/Sales": 1.1234,
    "Cost of Capital": 12.34,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 12.34,
    "Sales Cap Ratio": 12.34,
    "ROIC": 12.34,
    "Tax Rate": 12.34
  },
  "Advertising": {
    "EV/Sales": 2.12,
    "Cost of Capital": 7.81,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 10.05,
    "Sales Cap Ratio": 12.34,
    "ROIC": 12.34,
    "Tax Rate": 5.02,
    "Cost of Debt": 5.29
  },
  "Aerospace/Defense": {
    "EV/Sales": 3.57,
    "Cost of Capital": 7.6,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 8.7,
    "Sales Cap Ratio": 12.34,
    "ROIC": 12.34,
    "Tax Rate": 11.58,
    "Cost of Debt": 5.29
  },
  "Air Transport": {
    "EV/Sales": 1.03,
    "Cost of Capital": 6.72,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 4.88,
    "Sales Cap Ratio": 12.34,
    "ROIC": 12.34,
    "Tax Rate": 8.29,
    "Cost of Debt"

In [10]:
pbvdata_url = "https://pages.stern.nyu.edu/~adamodar/New_Home_Page/datafile/pbvdata.html"

try:
    response = requests.get(pbvdata_url)
    response.raise_for_status()

    pbvdata_soup = BeautifulSoup(response.text, 'html.parser')

    all_tables = pbvdata_soup.find_all('table')

    pbvdata_table = next(table for table in all_tables if 'Advertising' in str(table))
    if not pbvdata_table:
        raise ValueError("Could not find the pbvdata table in the HTML content.")
    
except Exception as e:
    print(f"Error fetching or parsing pbvdata data: {e}")
    pbvdata_table = None

pbvdata_rows = pbvdata_table.find_all('tr')[1:]
pbvdata_data = {}

for row in pbvdata_rows:
    cols = row.find_all('td')
    industry_name = ' '.join(cols[0].get_text(strip=True).split())
    try:
        roic = float(cols[5].get_text(strip=True).replace('%', ''))
    except Exception as e:
        print(f"Invalid ROIC value for industry {industry_name}: {e}")
        roic = None

    pbvdata_data[industry_name] = {
        "ROIC": roic
    }

print(json.dumps(pbvdata_data, indent=2))

Invalid ROIC value for industry Bank (Money Center): could not convert string to float: 'NA'
Invalid ROIC value for industry Banks (Regional): could not convert string to float: 'NA'
Invalid ROIC value for industry Brokerage & Investment Banking: could not convert string to float: 'NA'
Invalid ROIC value for industry Financial Svcs. (Non-bank & Insurance): could not convert string to float: 'NA'
Invalid ROIC value for industry : could not convert string to float: ''
{
  "Advertising": {
    "ROIC": 27.72
  },
  "Aerospace/Defense": {
    "ROIC": 16.01
  },
  "Air Transport": {
    "ROIC": 7.93
  },
  "Apparel": {
    "ROIC": 15.77
  },
  "Auto & Truck": {
    "ROIC": 2.25
  },
  "Auto Parts": {
    "ROIC": 8.98
  },
  "Bank (Money Center)": {
    "ROIC": null
  },
  "Banks (Regional)": {
    "ROIC": null
  },
  "Beverage (Alcoholic)": {
    "ROIC": 15.74
  },
  "Beverage (Soft)": {
    "ROIC": 29.03
  },
  "Broadcasting": {
    "ROIC": 14.57
  },
  "Brokerage & Investment Banking": {
 

In [11]:
for industry in pbvdata_data:
    if industry == '':
        continue
    if industry not in market_data['industries']:
        print(f"Adding industry '{industry}' to market data.")
        market_data['industries'][industry] = market_data['industries']['Choose an Industry'].copy()
    market_data['industries'][industry]['ROIC'] = pbvdata_data[industry]['ROIC']

print(json.dumps(market_data['industries'], indent=2))

{
  "Choose an Industry": {
    "EV/Sales": 1.1234,
    "Cost of Capital": 12.34,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 12.34,
    "Sales Cap Ratio": 12.34,
    "ROIC": 12.34,
    "Tax Rate": 12.34
  },
  "Advertising": {
    "EV/Sales": 2.12,
    "Cost of Capital": 7.81,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 10.05,
    "Sales Cap Ratio": 12.34,
    "ROIC": 27.72,
    "Tax Rate": 5.02,
    "Cost of Debt": 5.29
  },
  "Aerospace/Defense": {
    "EV/Sales": 3.57,
    "Cost of Capital": 7.6,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 8.7,
    "Sales Cap Ratio": 12.34,
    "ROIC": 16.01,
    "Tax Rate": 11.58,
    "Cost of Debt": 5.29
  },
  "Air Transport": {
    "EV/Sales": 1.03,
    "Cost of Capital": 6.72,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 4.88,
    "Sales Cap Ratio": 12.34,
    "ROIC": 7.93,
    "Tax Rate": 8.29,
    "Cost of Debt":

In [12]:
histgr_url = "https://pages.stern.nyu.edu/~adamodar/New_Home_Page/datafile/histgr.html"

try:
    response = requests.get(histgr_url)
    response.raise_for_status()

    histgr_soup = BeautifulSoup(response.text, 'html.parser')

    all_tables = histgr_soup.find_all('table')

    histgr_table = next(table for table in all_tables if 'Advertising' in str(table))
    if not histgr_table:
        raise ValueError("Could not find the histgr table in the HTML content.")
    
except Exception as e:
    print(f"Error fetching or parsing histgr data: {e}")
    histgr_table = None

histgr_rows = histgr_table.find_all('tr')[1:]
histgr_data = {}

for row in histgr_rows:
    cols = row.find_all('td')
    industry_name = ' '.join(cols[0].get_text(strip=True).split())
    try:
        rev_growth = float(cols[3].get_text(strip=True).replace('%', ''))
    except Exception as e:
        print(f"Invalid revenue growth value for industry {industry_name}: {e}")
        rev_growth = None

    histgr_data[industry_name] = {
        'Revenue Growth Rate': rev_growth
    }

print(json.dumps(histgr_data, indent=2))

Invalid revenue growth value for industry : could not convert string to float: ''
{
  "Advertising": {
    "Revenue Growth Rate": 17.67
  },
  "Aerospace/Defense": {
    "Revenue Growth Rate": 11.1
  },
  "Air Transport": {
    "Revenue Growth Rate": 47.79
  },
  "Apparel": {
    "Revenue Growth Rate": 8.1
  },
  "Auto & Truck": {
    "Revenue Growth Rate": 8.64
  },
  "Auto Parts": {
    "Revenue Growth Rate": 15.91
  },
  "Bank (Money Center)": {
    "Revenue Growth Rate": 9.06
  },
  "Banks (Regional)": {
    "Revenue Growth Rate": 8.05
  },
  "Beverage (Alcoholic)": {
    "Revenue Growth Rate": 3.43
  },
  "Beverage (Soft)": {
    "Revenue Growth Rate": 8.58
  },
  "Broadcasting": {
    "Revenue Growth Rate": 8.41
  },
  "Brokerage & Investment Banking": {
    "Revenue Growth Rate": 35.52
  },
  "Building Materials": {
    "Revenue Growth Rate": 4.14
  },
  "Business & Consumer Services": {
    "Revenue Growth Rate": 5.8
  },
  "Cable TV": {
    "Revenue Growth Rate": 26.94
  },
  

In [13]:
for industry in histgr_data:
    if industry == '':
        continue
    if industry not in market_data['industries']:
        print(f"Adding industry '{industry}' to market data.")
        market_data['industries'][industry] = market_data['industries']['Choose an Industry'].copy()
    market_data['industries'][industry]['Revenue Growth Rate'] = histgr_data[industry]['Revenue Growth Rate']

print(json.dumps(market_data['industries'], indent=2))

{
  "Choose an Industry": {
    "EV/Sales": 1.1234,
    "Cost of Capital": 12.34,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 12.34,
    "Sales Cap Ratio": 12.34,
    "ROIC": 12.34,
    "Tax Rate": 12.34
  },
  "Advertising": {
    "EV/Sales": 2.12,
    "Cost of Capital": 7.81,
    "Beta": 12.34,
    "Revenue Growth Rate": 17.67,
    "Operating Margin": 10.05,
    "Sales Cap Ratio": 12.34,
    "ROIC": 27.72,
    "Tax Rate": 5.02,
    "Cost of Debt": 5.29
  },
  "Aerospace/Defense": {
    "EV/Sales": 3.57,
    "Cost of Capital": 7.6,
    "Beta": 12.34,
    "Revenue Growth Rate": 11.1,
    "Operating Margin": 8.7,
    "Sales Cap Ratio": 12.34,
    "ROIC": 16.01,
    "Tax Rate": 11.58,
    "Cost of Debt": 5.29
  },
  "Air Transport": {
    "EV/Sales": 1.03,
    "Cost of Capital": 6.72,
    "Beta": 12.34,
    "Revenue Growth Rate": 47.79,
    "Operating Margin": 4.88,
    "Sales Cap Ratio": 12.34,
    "ROIC": 7.93,
    "Tax Rate": 8.29,
    "Cost of Debt": 

In [14]:
capex_url = "https://pages.stern.nyu.edu/~adamodar/New_Home_Page/datafile/capex.html"

try:
    response = requests.get(capex_url)
    response.raise_for_status()

    capex_soup = BeautifulSoup(response.text, 'html.parser')

    all_tables = capex_soup.find_all('table')

    capex_table = next(table for table in all_tables if 'Advertising' in str(table))
    if not capex_table:
        raise ValueError("Could not find the capex table in the HTML content.")
    
except Exception as e:
    print(f"Error fetching or parsing capex data: {e}")
    capex_table = None

capex_rows = capex_table.find_all('tr')[1:]
capex_data = {}

for row in capex_rows:
    cols = row.find_all('td')
    industry_name = ' '.join(cols[0].get_text(strip=True).split())
    try:
        sales_cap = float(cols[9].get_text(strip=True).replace('%', ''))
    except Exception as e:
        print(f"Invalid sales cap value for industry {industry_name}: {e}")
        sales_cap = None

    capex_data[industry_name] = {
        'Sales Cap Ratio': sales_cap
    }

print(json.dumps(capex_data, indent=2))

Invalid sales cap value for industry : could not convert string to float: ''
{
  "Advertising": {
    "Sales Cap Ratio": 3.85
  },
  "Aerospace/Defense": {
    "Sales Cap Ratio": 2.74
  },
  "Air Transport": {
    "Sales Cap Ratio": 1.8
  },
  "Apparel": {
    "Sales Cap Ratio": 1.77
  },
  "Auto & Truck": {
    "Sales Cap Ratio": 1.08
  },
  "Auto Parts": {
    "Sales Cap Ratio": 2.39
  },
  "Bank (Money Center)": {
    "Sales Cap Ratio": 0.19
  },
  "Banks (Regional)": {
    "Sales Cap Ratio": 0.33
  },
  "Beverage (Alcoholic)": {
    "Sales Cap Ratio": 0.79
  },
  "Beverage (Soft)": {
    "Sales Cap Ratio": 1.54
  },
  "Broadcasting": {
    "Sales Cap Ratio": 1.28
  },
  "Brokerage & Investment Banking": {
    "Sales Cap Ratio": 0.27
  },
  "Building Materials": {
    "Sales Cap Ratio": 2.06
  },
  "Business & Consumer Services": {
    "Sales Cap Ratio": 2.8
  },
  "Cable TV": {
    "Sales Cap Ratio": 0.74
  },
  "Chemical (Basic)": {
    "Sales Cap Ratio": 1.49
  },
  "Chemical (Di

In [15]:
for industry in capex_data:
    if industry == '':
        continue
    if industry not in market_data['industries']:
        print(f"Adding industry '{industry}' to market data.")
        market_data['industries'][industry] = market_data['industries']['Choose an Industry'].copy()
    market_data['industries'][industry]['Sales Cap Ratio'] = capex_data[industry]['Sales Cap Ratio']

print(json.dumps(market_data['industries'], indent=2))

{
  "Choose an Industry": {
    "EV/Sales": 1.1234,
    "Cost of Capital": 12.34,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 12.34,
    "Sales Cap Ratio": 12.34,
    "ROIC": 12.34,
    "Tax Rate": 12.34
  },
  "Advertising": {
    "EV/Sales": 2.12,
    "Cost of Capital": 7.81,
    "Beta": 12.34,
    "Revenue Growth Rate": 17.67,
    "Operating Margin": 10.05,
    "Sales Cap Ratio": 3.85,
    "ROIC": 27.72,
    "Tax Rate": 5.02,
    "Cost of Debt": 5.29
  },
  "Aerospace/Defense": {
    "EV/Sales": 3.57,
    "Cost of Capital": 7.6,
    "Beta": 12.34,
    "Revenue Growth Rate": 11.1,
    "Operating Margin": 8.7,
    "Sales Cap Ratio": 2.74,
    "ROIC": 16.01,
    "Tax Rate": 11.58,
    "Cost of Debt": 5.29
  },
  "Air Transport": {
    "EV/Sales": 1.03,
    "Cost of Capital": 6.72,
    "Beta": 12.34,
    "Revenue Growth Rate": 47.79,
    "Operating Margin": 4.88,
    "Sales Cap Ratio": 1.8,
    "ROIC": 7.93,
    "Tax Rate": 8.29,
    "Cost of Debt": 5.29

In [16]:
# NB: we are gathering the UNLEVERED BETA, not beta (which is levered)
beta_url = "https://pages.stern.nyu.edu/~adamodar/New_Home_Page/datafile/Betas.html"

try:
    response = requests.get(beta_url)
    response.raise_for_status()

    beta_soup = BeautifulSoup(response.text, 'html.parser')

    all_tables = beta_soup.find_all('table')

    beta_table = next(table for table in all_tables if 'Advertising' in str(table))
    if not beta_table:
        raise ValueError("Could not find the beta table in the HTML content.")
    
except Exception as e:
    print(f"Error fetching or parsing beta data: {e}")
    beta_table = None

beta_rows = beta_table.find_all('tr')[1:]
beta_data = {}

for row in beta_rows:
    cols = row.find_all('td')
    industry_name = ' '.join(cols[0].get_text(strip=True).split())
    try:
        beta = float(cols[5].get_text(strip=True).replace('%', ''))
    except Exception as e:
        print(f"Invalid beta value for industry {industry_name}: {e}")
        beta = None

    beta_data[industry_name] = {
        'Beta': beta
    }

print(json.dumps(beta_data, indent=2))

Invalid beta value for industry : could not convert string to float: ''
{
  "Advertising": {
    "Beta": 0.93
  },
  "Aerospace/Defense": {
    "Beta": 0.85
  },
  "Air Transport": {
    "Beta": 0.7
  },
  "Apparel": {
    "Beta": 0.76
  },
  "Auto & Truck": {
    "Beta": 1.27
  },
  "Auto Parts": {
    "Beta": 1.02
  },
  "Bank (Money Center)": {
    "Beta": 0.34
  },
  "Banks (Regional)": {
    "Beta": 0.29
  },
  "Beverage (Alcoholic)": {
    "Beta": 0.61
  },
  "Beverage (Soft)": {
    "Beta": 0.56
  },
  "Broadcasting": {
    "Beta": 0.29
  },
  "Brokerage & Investment Banking": {
    "Beta": 0.58
  },
  "Building Materials": {
    "Beta": 0.93
  },
  "Business & Consumer Services": {
    "Beta": 0.77
  },
  "Cable TV": {
    "Beta": 0.35
  },
  "Chemical (Basic)": {
    "Beta": 0.58
  },
  "Chemical (Diversified)": {
    "Beta": 0.37
  },
  "Chemical (Specialty)": {
    "Beta": 0.79
  },
  "Coal & Related Energy": {
    "Beta": 1.02
  },
  "Computer Services": {
    "Beta": 0.92


In [17]:
for industry in beta_data:
    if industry == '':
        continue
    if industry not in market_data['industries']:
        print(f"Adding industry '{industry}' to market data.")
        market_data['industries'][industry] = market_data['industries']['Choose an Industry'].copy()
    market_data['industries'][industry]['Beta'] = beta_data[industry]['Beta']

print(json.dumps(market_data['industries'], indent=2))

{
  "Choose an Industry": {
    "EV/Sales": 1.1234,
    "Cost of Capital": 12.34,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 12.34,
    "Sales Cap Ratio": 12.34,
    "ROIC": 12.34,
    "Tax Rate": 12.34
  },
  "Advertising": {
    "EV/Sales": 2.12,
    "Cost of Capital": 7.81,
    "Beta": 0.93,
    "Revenue Growth Rate": 17.67,
    "Operating Margin": 10.05,
    "Sales Cap Ratio": 3.85,
    "ROIC": 27.72,
    "Tax Rate": 5.02,
    "Cost of Debt": 5.29
  },
  "Aerospace/Defense": {
    "EV/Sales": 3.57,
    "Cost of Capital": 7.6,
    "Beta": 0.85,
    "Revenue Growth Rate": 11.1,
    "Operating Margin": 8.7,
    "Sales Cap Ratio": 2.74,
    "ROIC": 16.01,
    "Tax Rate": 11.58,
    "Cost of Debt": 5.29
  },
  "Air Transport": {
    "EV/Sales": 1.03,
    "Cost of Capital": 6.72,
    "Beta": 0.7,
    "Revenue Growth Rate": 47.79,
    "Operating Margin": 4.88,
    "Sales Cap Ratio": 1.8,
    "ROIC": 7.93,
    "Tax Rate": 8.29,
    "Cost of Debt": 5.29
  }

In [18]:
ratings_url = "https://pages.stern.nyu.edu/~adamodar/New_Home_Page/datafile/ratings.html"

try:
    response = requests.get(ratings_url)
    response.raise_for_status()

    ratings_soup = BeautifulSoup(response.text, 'html.parser')

    all_tables = ratings_soup.find_all('table')

    ratings_table = all_tables[0]

    if not ratings_table:
        raise ValueError("Could not find the ratings table in the HTML content.")
    
except Exception as e:
    print(f"Error fetching or parsing ratings data: {e}")
    all_tables = None

ratings_rows = ratings_table.find_all('tr')[1:]
pruned_ratings_rows = []
ratings_data = {}

for row in ratings_rows:
    cols = row.find_all('td')
    if cols[2].getText(strip=True):
        if cols[2].getText(strip=True) == "Rating is":
            continue
        pruned_ratings_rows.append([
            cols[0].get_text(strip=True),
            cols[1].get_text(strip=True),
            cols[2].get_text(strip=True),
            cols[3].get_text(strip=True).replace('%', '')
        ])

large_ratings = pruned_ratings_rows[:15]
small_ratings = pruned_ratings_rows[15:]

for [gt, lt, rating, spread] in large_ratings:
    ratings_data[rating] = {}
    try:
        ratings_data[rating]['gt_safe'] = float(gt)
        ratings_data[rating]['lt_safe'] = float(lt)
        ratings_data[rating]['Spread'] = float(spread)
    except Exception as e:
        print(f"Invalid rating bounds for rating {rating}: {e}")
        ratings_data[rating]['gt_safe'] = None
        ratings_data[rating]['lt_safe'] = None
        ratings_data[rating]['Spread'] = None

for [gt, lt, rating, spread] in small_ratings:
    try:
        ratings_data[rating]['gt_risk'] = float(gt)
        ratings_data[rating]['lt_risk'] = float(lt)
    except Exception as e:
        print(f"Invalid rating bounds for rating {rating}: {e}")
        ratings_data[rating]['gt_risk'] = None
        ratings_data[rating]['lt_risk'] = None

# For 2026, Damodaran did not report the risky firms default spread table. There is info on his ginzu model so I'll hardcode that for now.
ratings_data['D2/D']['gt_risk'] = float(-100000)
ratings_data['D2/D']['lt_risk'] = 0.5
ratings_data['C2/C']['gt_risk'] = 0.5
ratings_data['C2/C']['lt_risk'] = 0.8
ratings_data['Ca2/CC']['gt_risk'] = 0.8
ratings_data['Ca2/CC']['lt_risk'] = 1.25
ratings_data['Caa/CCC']['gt_risk'] = 1.25
ratings_data['Caa/CCC']['lt_risk'] = 1.5
ratings_data['B3/B-']['gt_risk'] = 1.5
ratings_data['B3/B-']['lt_risk'] = 2.0
ratings_data['B2/B']['gt_risk'] = 2.0
ratings_data['B2/B']['lt_risk'] = 2.5
ratings_data['B1/B+']['gt_risk'] = 2.5
ratings_data['B1/B+']['lt_risk'] = 3.0
ratings_data['Ba2/BB']['gt_risk'] = 3.0
ratings_data['Ba2/BB']['lt_risk'] = 3.5
ratings_data['Ba1/BB+']['gt_risk'] = 3.5
ratings_data['Ba1/BB+']['lt_risk'] = 4.0
ratings_data['Baa2/BBB']['gt_risk'] = 4.0
ratings_data['Baa2/BBB']['lt_risk'] = 4.5
ratings_data['A3/A-']['gt_risk'] = 4.5
ratings_data['A3/A-']['lt_risk'] = 6.0
ratings_data['A2/A']['gt_risk'] = 6.0
ratings_data['A2/A']['lt_risk'] = 7.5
ratings_data['A1/A+']['gt_risk'] = 7.5
ratings_data['A1/A+']['lt_risk'] = 9.5
ratings_data['Aa2/AA']['gt_risk'] = 9.5
ratings_data['Aa2/AA']['lt_risk'] = 12.5
ratings_data['Aaa/AAA']['gt_risk'] = 12.5
ratings_data['Aaa/AAA']['lt_risk'] = float(100000)

print(json.dumps(ratings_data, indent=2))

{
  "D2/D": {
    "gt_safe": -100000.0,
    "lt_safe": 0.199999,
    "Spread": 19.0,
    "gt_risk": -100000.0,
    "lt_risk": 0.5
  },
  "C2/C": {
    "gt_safe": 0.2,
    "lt_safe": 0.649999,
    "Spread": 16.0,
    "gt_risk": 0.5,
    "lt_risk": 0.8
  },
  "Ca2/CC": {
    "gt_safe": 0.65,
    "lt_safe": 0.799999,
    "Spread": 12.61,
    "gt_risk": 0.8,
    "lt_risk": 1.25
  },
  "Caa/CCC": {
    "gt_safe": 0.8,
    "lt_safe": 1.249999,
    "Spread": 8.85,
    "gt_risk": 1.25,
    "lt_risk": 1.5
  },
  "B3/B-": {
    "gt_safe": 1.25,
    "lt_safe": 1.499999,
    "Spread": 5.09,
    "gt_risk": 1.5,
    "lt_risk": 2.0
  },
  "B2/B": {
    "gt_safe": 1.5,
    "lt_safe": 1.749999,
    "Spread": 3.21,
    "gt_risk": 2.0,
    "lt_risk": 2.5
  },
  "B1/B+": {
    "gt_safe": 1.75,
    "lt_safe": 1.999999,
    "Spread": 2.75,
    "gt_risk": 2.5,
    "lt_risk": 3.0
  },
  "Ba2/BB": {
    "gt_safe": 2.0,
    "lt_safe": 2.2499999,
    "Spread": 1.84,
    "gt_risk": 3.0,
    "lt_risk": 3.5
  },
  

In [19]:
for rating in ratings_data:
    if rating == '':
        continue
    if rating not in market_data['credit_ratings']:
        print(f"Adding rating '{rating}' to market data.")
        market_data['credit_ratings'][rating] = market_data['credit_ratings']['Select a Credit Rating'].copy()
    market_data['credit_ratings'][rating]['gt_safe'] = ratings_data[rating]['gt_safe']
    market_data['credit_ratings'][rating]['lt_safe'] = ratings_data[rating]['lt_safe']
    market_data['credit_ratings'][rating]['gt_risk'] = ratings_data[rating]['gt_risk']
    market_data['credit_ratings'][rating]['lt_risk'] = ratings_data[rating]['lt_risk']
    market_data['credit_ratings'][rating]['Spread'] = ratings_data[rating]['Spread']

print(json.dumps(market_data['credit_ratings'], indent=2))

Adding rating 'D2/D' to market data.
Adding rating 'C2/C' to market data.
Adding rating 'Ca2/CC' to market data.
Adding rating 'Caa/CCC' to market data.
Adding rating 'B3/B-' to market data.
Adding rating 'B2/B' to market data.
Adding rating 'B1/B+' to market data.
Adding rating 'Ba2/BB' to market data.
Adding rating 'Ba1/BB+' to market data.
Adding rating 'Baa2/BBB' to market data.
Adding rating 'A3/A-' to market data.
Adding rating 'A2/A' to market data.
Adding rating 'A1/A+' to market data.
Adding rating 'Aa2/AA' to market data.
Adding rating 'Aaa/AAA' to market data.
{
  "Select a Credit Rating": {
    "Spread": 12.34,
    "gt_safe": 100001,
    "gt_risk": 100001,
    "lt_safe": -100001,
    "lt_risk": -100001
  },
  "D2/D": {
    "Spread": 19.0,
    "gt_safe": -100000.0,
    "gt_risk": -100000.0,
    "lt_safe": 0.199999,
    "lt_risk": 0.5
  },
  "C2/C": {
    "Spread": 16.0,
    "gt_safe": 0.2,
    "gt_risk": 0.5,
    "lt_safe": 0.649999,
    "lt_risk": 0.8
  },
  "Ca2/CC": {
   

In [20]:
# last item, regional erp seems to only be saved on an xlsx file
# will parse this data using pandas instead
ctryprem_xlsx_url = "https://www.stern.nyu.edu/~adamodar/pc/datasets/ctryprem.xlsx"
try:
    response = requests.get(ctryprem_xlsx_url)
    response.raise_for_status()

    ctryprem_xlsx_file = BytesIO(response.content)

    df = pd.read_excel(ctryprem_xlsx_file, sheet_name='Regional Weighted Averages')

    emea_erp = df[df.iloc[:, 0] == 'EMEA'].iloc[0, 1]

    ctryprem_df = df.iloc[169:179, :2]
    ctryprem_df.columns = ['Region', 'ERP']
    ctryprem_df = ctryprem_df.set_index('Region')

except Exception as e:
    print(f"Error fetching or parsing country premium xlsx data: {e}")

print('EMEA ERP: ', emea_erp)
print(ctryprem_df)

EMEA ERP:  0.0591478631288238
                                ERP
Region                             
Africa                     0.119493
Asia                       0.057223
Australia & New Zealand    0.042341
Caribbean                  0.117177
Central and South America  0.084628
Eastern Europe             0.075777
Middle East                0.062367
North America              0.044465
Western Europe             0.052665
Global                     0.056257


In [21]:
for region in ctryprem_df.index:
    if region == '':
        continue
    if region not in market_data['regions']:
        print(f"Adding region '{region}' to market data.")
        market_data['regions'][region] = market_data['regions']['Select a Region'].copy()
    market_data['regions'][region]['ERP'] = ctryprem_df.loc[region, 'ERP'] * 100

print('Manually adding Rest of World as Global ERP.')
market_data['regions']['Rest of World'] = market_data['regions']['Select a Region'].copy()
market_data['regions']['Rest of World']['ERP'] = ctryprem_df.loc['Global', 'ERP'] * 100

print(json.dumps(market_data['regions'], indent=2))

Adding region 'Africa' to market data.
Adding region 'Asia' to market data.
Adding region 'Australia & New Zealand' to market data.
Adding region 'Caribbean' to market data.
Adding region 'Central and South America' to market data.
Adding region 'Eastern Europe' to market data.
Adding region 'Middle East' to market data.
Adding region 'North America' to market data.
Adding region 'Western Europe' to market data.
Adding region 'Global' to market data.
Manually adding Rest of World as Global ERP.
{
  "Select a Region": {
    "ERP": 12.34
  },
  "Africa": {
    "ERP": 11.949308050536839
  },
  "Asia": {
    "ERP": 5.722323564845226
  },
  "Australia & New Zealand": {
    "ERP": 4.234082173200422
  },
  "Caribbean": {
    "ERP": 11.7176823349771
  },
  "Central and South America": {
    "ERP": 8.462849620618984
  },
  "Eastern Europe": {
    "ERP": 7.577709101845076
  },
  "Middle East": {
    "ERP": 6.2366864403681905
  },
  "North America": {
    "ERP": 4.446474878479405
  },
  "Western 

In [22]:
# Market data should be fully populated now, write to file in FrontEnd
try:
    with open('../FrontEnd/src/utils/marketData.json', 'w') as f:
        json.dump(market_data, f, indent=2)
        print("Market data successfully written to marketData.json")
except Exception as e:
    print(f"Error writing market data to file: {e}")

Market data successfully written to marketData.json
